# Day 03 — Functions + Data Structures

======================================
Module 01: Python + Async + FastAPI for LLM Engineering
Vidya Sankalp | Applied GenAI Engineering

WHAT THIS FILE COVERS:
  - def, parameters, return, default values, docstrings
  - *args and **kwargs — flexible function signatures
  - lambda — one-line functional transforms
  - List: message history, search results
  - Dict: model config, structured LLM output
  - Set: unique tool names, deduplication
  - Tuple: immutable pairs (key, value)

WHY THIS MATTERS FOR LLM ENGINEERING:
  - LLM message history IS a list of dicts
  - Model configuration IS a dict
  - Tool names in an agent IS a set (unique, fast lookup)
  - Functions wrap every prompt-building step


In [ ]:

from typing import Any   # for type hints when the type varies




## SECTION 1: FUNCTIONS

═════════════════════════════════════════════════════════════════════════════


In [ ]:

def build_system_prompt(
    role: str,
    domain: str,
    rules: list[str],
    output_format: str = "plain text",  # default value — caller can override
) -> str:
    """
    Build a structured 5-section system prompt.

    This implements the prompt anatomy taught in Module 03 (Prompt Engineering).
    The system prompt is a fixed string that configures the LLM's behaviour
    for the entire conversation. It does NOT change per request.

    Args:
        role         : Who the LLM is (e.g. "customer support assistant")
        domain       : The business domain (e.g. "e-commerce platform ShopSmart")
        rules        : List of MUST/MUST NOT rules for the LLM to follow
        output_format: How the response should be formatted (default: plain text)

    Returns:
        A complete system prompt string ready to send to the LLM API.
    """

    # Build the rules block — each rule on its own line with bullet prefix
    rules_block = "\n".join(f"- {rule}" for rule in rules)

    # Triple-quote f-string: newlines and indentation are preserved exactly
    # This is a direct implementation of the 5-section template from Module 03
    prompt = f"""## Role
You are a {role} for {domain}.

## Context
You help customers with product questions, order tracking, returns, and general support.
All information you provide must come from the data given to you — never invent details.

## Task
Answer the customer's question clearly and accurately.
{rules_block}

## Output Format
{output_format}

## Examples
Input : "Where is order #3042?"
Output: "Order #3042 is currently In Transit and expected by Friday."
"""
    return prompt



In [ ]:
def build_user_message(
    query: str,
    context: dict | None = None,  # optional — not all messages have DB context
) -> str:
    """
    Build the user-turn message, optionally injecting database context.

    The user message is sent on EVERY request (unlike the system prompt
    which is fixed). It contains the customer's actual question plus
    any relevant data retrieved from the database (orders, products etc.)

    Args:
        query  : The raw text from the customer.
        context: Optional dict of relevant data from the database.

    Returns:
        A formatted user message string.
    """

    if context is None:
        # Simple query — no database lookup needed
        return f"<message>{query}</message>"

    # Build a context block from the dict
    # The XML tag <context> tells the LLM this is retrieved data, not user input
    context_lines = [f"  {k}: {v}" for k, v in context.items()]
    context_block = "\n".join(context_lines)

    return (
        f"<context>\n{context_block}\n</context>\n\n"
        f"<message>{query}</message>"
    )



In [ ]:
def validate_prompt_length(
    prompt: str,
    max_chars: int = 4000,
    *,                        # force all arguments after * to be keyword-only
    warn_at: int = 3000,      # keyword-only: caller MUST write warn_at=3000
) -> tuple[bool, str]:
    """
    Check if a prompt is within acceptable length limits.

    Returns a (is_valid, message) tuple — Python functions can return
    multiple values by returning a tuple.

    Args:
        prompt  : The prompt string to check.
        max_chars: Hard limit — reject the prompt above this.
        warn_at : Soft limit — warn but still allow.

    Returns:
        (True, "ok") if valid
        (True, "warning: ...") if long but still allowed
        (False, "error: ...") if over the hard limit
    """

    length = len(prompt)

    if length > max_chars:
        return False, f"error: prompt is {length} chars, limit is {max_chars}"
    elif length > warn_at:
        return True, f"warning: prompt is {length} chars, consider trimming"
    else:
        return True, f"ok: {length} chars"




In [ ]:

def combine_context_chunks(*chunks: str, separator: str = "\n\n") -> str:
    """
    Combine multiple retrieved context chunks into one string.

    *chunks means the caller can pass any number of string arguments.
    This is useful when assembling RAG context from multiple retrieved documents.

    Args:
        *chunks  : Any number of context string chunks.
        separator: String placed between each chunk.

    Returns:
        All chunks joined into a single string.

    Example:
        combine_context_chunks(chunk1, chunk2, chunk3)
        # → "chunk1\n\nchunk2\n\nchunk3"
    """

    # chunks is a tuple of strings inside the function
    # Filter out empty strings before joining
    non_empty = [c.strip() for c in chunks if c.strip()]
    return separator.join(non_empty)




In [ ]:

def build_api_request(
    model: str,
    messages: list[dict],
    **kwargs: Any,             # capture any extra keyword arguments
) -> dict:
    """
    Build an OpenAI-compatible API request payload.

    **kwargs lets the caller pass extra parameters (temperature, max_tokens,
    stream, etc.) without this function needing to know about all of them.
    This is the real reason **kwargs exists in LLM engineering code.

    Args:
        model   : Model identifier (e.g. "gpt-4o")
        messages: Conversation history list
        **kwargs: Any additional OpenAI API parameters

    Returns:
        A dict that can be sent directly to the OpenAI API.

    Example:
        payload = build_api_request(
            model="gpt-4o",
            messages=[{"role": "user", "content": "Hello"}],
            temperature=0.2,
            max_tokens=500,
            stream=True,
        )
    """

    # Start with the required parameters
    payload: dict = {
        "model"   : model,
        "messages": messages,
    }

    # Merge any extra kwargs into the payload
    # This is equivalent to: payload.update(kwargs)
    payload = {**payload, **kwargs}   # dict unpacking — merges two dicts

    return payload




In [ ]:

# Lambda is useful for short transforms — not for complex logic
# Rule of thumb: if it doesn't fit on one line, write a full def instead

# Strip whitespace from both ends (common when reading CSV data)
clean_text = lambda text: text.strip()

# Convert a rating int (1-5) to a star string "★★★☆☆"
rating_to_stars = lambda r: "★" * r + "☆" * (5 - r)

# Truncate a product description for use in a prompt (avoid context overflow)
truncate = lambda text, n=200: text[:n] + "..." if len(text) > n else text




## SECTION 2: DATA STRUCTURES IN LLM ENGINEERING CONTEXT

═════════════════════════════════════════════════════════════════════════════


## The LLM API expects a list of dicts, one per turn in the conversation.

This is not an abstraction — this IS the actual OpenAI API format.


In [ ]:

def build_conversation_history(
    system_prompt: str,
    turns: list[tuple[str, str]],  # list of (user_msg, assistant_msg) pairs
) -> list[dict]:
    """
    Build a conversation history list in OpenAI message format.

    Args:
        system_prompt: The fixed system message.
        turns        : List of (user_message, assistant_response) tuples.

    Returns:
        A list of {"role": ..., "content": ...} dicts.
    """

    # Start with the system message
    messages: list[dict] = [
        {"role": "system", "content": system_prompt}
    ]

    # Add each turn as a user + assistant pair
    for user_msg, assistant_msg in turns:
        messages.append({"role": "user",      "content": user_msg})
        messages.append({"role": "assistant", "content": assistant_msg})

    return messages



In [ ]:
def add_user_turn(messages: list[dict], user_message: str) -> list[dict]:
    """
    Add the latest user message to an existing conversation history.

    Args:
        messages    : Existing conversation history list.
        user_message: The new user message to append.

    Returns:
        The updated messages list (same list, mutated in place).
    """

    messages.append({"role": "user", "content": user_message})
    return messages




In [ ]:

# A dict is the natural Python representation of a JSON config object.
# This model_config dict is passed as **kwargs to build_api_request() above.
model_config: dict = {
    "temperature" : 0.2,      # low = consistent, factual responses
    "max_tokens"  : 512,      # short enough for support chat
    "top_p"       : 0.9,
    "stream"      : False,    # set True in FastAPI streaming endpoint (Day 13)
}



In [ ]:
def get_model_config(override: dict | None = None) -> dict:
    """
    Return the model config, optionally overriding specific keys.

    This pattern (default config + caller overrides) is used throughout
    LangChain and LiteLLM to configure LLM calls.

    Args:
        override: Dict of keys to override in the default config.

    Returns:
        Merged configuration dict.
    """

    if override is None:
        return model_config.copy()   # .copy() prevents mutation of the original

    # Merge: start with defaults, then apply overrides
    # The order matters: override values win because they come second
    return {**model_config, **override}




In [ ]:

# Sets store only unique values and support O(1) membership testing (fast!).
# In an agent, the set of available tool names is checked on every LLM response.

AVAILABLE_TOOLS: set[str] = {
    "lookup_order",
    "lookup_customer",
    "lookup_product",
    "check_inventory",
    "search_reviews",
}



In [ ]:
def is_valid_tool(tool_name: str) -> bool:
    """
    Check if a tool name is in the allowed set.

    Using a set for this check is O(1) — it doesn't slow down
    as AVAILABLE_TOOLS grows, unlike a list which is O(n).

    Args:
        tool_name: The tool the LLM wants to call.

    Returns:
        True if the tool exists and can be called.
    """

    return tool_name in AVAILABLE_TOOLS   # set membership: O(1)




In [ ]:

# Tuples are immutable — once created, values cannot be changed.
# Use tuples when the pair of values should NOT be modified.

# A prompt version is a (version_number, description) pair.
# We don't want anyone accidentally modifying the version record.
PROMPT_VERSIONS: list[tuple[int, str]] = [
    (1, "Initial customer support prompt"),
    (2, "Added refund handling rules"),
    (3, "Improved output format spec"),
    (4, "Added few-shot examples"),
]



In [ ]:
def get_latest_prompt_version() -> tuple[int, str]:
    """Return the most recent (version, description) tuple."""
    return PROMPT_VERSIONS[-1]   # last item in list




In [ ]:
if __name__ == "__main__":
    print("=" * 60)
    print("DAY 03 — Functions + Data Structures")
    print("=" * 60)

    # Functions
    print("\n[1] System Prompt")
    sys_prompt = build_system_prompt(
        role="customer support specialist",
        domain="ShopSmart e-commerce platform",
        rules=[
            "You MUST only use information provided in the context",
            "You MUST NOT make up order details or prices",
            "You MUST be polite even if the customer is frustrated",
        ],
        output_format="2-3 sentences maximum",
    )
    print(sys_prompt[:300] + "...")

    print("\n[2] User Message with Context")
    ctx = {"order_id": "3042", "status": "In Transit", "product": "Classic Monitor"}
    msg = build_user_message("Where is my order?", context=ctx)
    print(msg)

    print("\n[3] Prompt length validation")
    ok, info = validate_prompt_length(sys_prompt, warn_at=300)
    print(f"  Valid: {ok}  |  {info}")

    # Data structures
    print("\n[4] Conversation History (list of dicts)")
    history = build_conversation_history(
        system_prompt="You are a support agent.",
        turns=[
            ("What is your return policy?", "Returns accepted within 30 days."),
        ],
    )
    for msg in history:
        print(f"  role={msg['role']:10s} | {msg['content'][:50]}")

    print("\n[5] Model Config (dict with override)")
    config = get_model_config(override={"temperature": 0.0, "stream": True})
    print(f"  {config}")

    print("\n[6] Tool validation (set)")
    for tool in ["lookup_order", "delete_database", "search_reviews"]:
        valid = is_valid_tool(tool)
        print(f"  '{tool}': {'VALID' if valid else 'REJECTED'}")

    print("\n[7] Latest prompt version (tuple)")
    ver, desc = get_latest_prompt_version()
    print(f"  v{ver}: {desc}")

    print("\n[8] Lambda transforms")
    print(f"  rating_to_stars(4)    : {rating_to_stars(4)}")
    print(f"  truncate(long_text)   : {truncate('This is a very long product description that needs to be shortened', 30)}")


## Run the demo

Run the cell below to execute the `if __name__ == '__main__':` block.


In [ ]:
# Execute the demo for this day
import runpy
runpy.run_path('modules/day03_functions_data_structures.py', run_name='__main__')
